# fMRI · 뇌 영역의 신호는 함께 움직일까? (합성 데이터 예제)

이 노트북은 **연습용 합성 데이터**를 사용합니다. 실제 사람의 뇌 데이터가 아니며, 세 영역 A, B, C의 관계를 만든 사람이 정답으로 알고 있습니다.

먼저 같은 폴더에서 데이터를 만듭니다 (약 20초).

```bash
python generate_synthetic_fmri.py
```

`data/ground_truth.json`에 정답이 들어 있습니다. **마지막 단계 전까지 열어 보지 마세요.**

In [ ]:
import json
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn.maskers import NiftiLabelsMasker

bold = nib.load("data/bold.nii.gz")
rois = nib.load("data/rois.nii.gz")
confounds = pd.read_csv("data/confounds.tsv", sep="\t")
tr = float(bold.header.get_zooms()[3])
print("볼륨 크기 (x, y, z, 시간):", bold.shape, "| TR:", tr, "초")
print("영역 라벨:", np.unique(rois.get_fdata()).astype(int), "(0=배경, 1=A, 2=B, 3=C)")

## 1. 데이터 살펴보기

fMRI는 뇌를 작은 정육면체(복셀)로 나누고, 몇 초 간격(TR)으로 각 복셀의 신호를 반복해서 찍습니다. 한 시점의 단면과 세 영역의 위치를 그려 봅니다.

In [ ]:
data, labels = bold.get_fdata(), rois.get_fdata()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (k, name) in zip(axes, enumerate("ABC", start=1)):
    z = int(np.round(np.argwhere(labels == k)[:, 2].mean()))
    ax.imshow(data[:, :, z, 0].T, cmap="gray", origin="lower")
    ax.contour((labels[:, :, z] == k).T, levels=[0.5], colors="red")
    ax.set_title(f"ROI {name} (z={z})"); ax.axis("off")
plt.tight_layout()

## 2. 영역별 시계열 뽑기

영역 안 복셀들의 신호를 평균해 영역마다 시계열 하나를 만듭니다. `detrend`와 `high_pass`는 스캐너에서 생기는 아주 느린 신호 변화(drift)를 제거합니다.

In [ ]:
masker = NiftiLabelsMasker(rois, detrend=True, high_pass=0.01, t_r=tr,
                           standardize="zscore_sample")
ts = masker.fit_transform(bold)          # (시간, 영역)
time = np.arange(ts.shape[0]) * tr
fig, ax = plt.subplots(figsize=(12, 4))
for k, name in enumerate("ABC"):
    z = (ts[:, k] - ts[:, k].mean()) / ts[:, k].std()
    ax.plot(time, z + 4 * k, label=f"ROI {name}")
ax.set_xlabel("time (s)"); ax.set_yticks([]); ax.legend(loc="upper right")
ax.set_title("ROI signals (z-scored, vertically offset)");

## 3. 상관계수 계산

상관계수는 두 신호가 함께 오르내리는 정도입니다 (−1 ~ 1).

In [ ]:
def pairwise(ts):
    c = np.corrcoef(ts.T)
    return pd.Series({"A-B": c[0, 1], "A-C": c[0, 2], "B-C": c[1, 2]}).round(2)

before = pairwise(ts)
before

**생각해 보기:** 세 쌍 모두 양의 상관이 보입니다. 정말 세 영역이 모두 함께 움직이는 걸까요? 다른 설명은 없을까요?

## 4. 머리 움직임 확인

`framewise_displacement`(FD)는 직전 시점과 비교한 머리 움직임의 크기(mm)입니다. 머리가 크게 움직인 순간에는 뇌 전체의 신호가 한꺼번에 바뀌어, 서로 관계없는 영역도 함께 움직이는 것처럼 보일 수 있습니다.

In [ ]:
fd = confounds["framewise_displacement"].to_numpy()
threshold = 0.5  # mm
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(time, fd, color="k"); axes[0].axhline(threshold, color="r", ls="--")
axes[0].set_ylabel("FD (mm)")
for k, name in enumerate("ABC"):
    axes[1].plot(time, (ts[:, k] - ts[:, k].mean()) / ts[:, k].std(), label=f"ROI {name}", alpha=0.8)
for t in time[fd > threshold]:
    axes[1].axvline(t, color="r", alpha=0.2)
axes[1].set_xlabel("time (s)"); axes[1].legend(loc="upper right")
print("FD >", threshold, "mm 인 시점 수:", int((fd > threshold).sum()))

## 5. 움직임이 큰 시점을 빼고 다시 계산 (scrubbing)

FD가 기준보다 큰 시점과 그 바로 다음 시점을 제외합니다.

In [ ]:
keep = fd <= threshold
keep[1:] &= fd[:-1] <= threshold
after = pairwise(ts[keep])
print(f"사용한 시점: {keep.sum()} / {len(keep)}")
pd.DataFrame({"움직임 제거 전": before, "움직임 제거 후": after})

## 6. 정답과 비교

이제 `ground_truth.json`을 엽니다. 정답은 잡음과 움직임을 넣기 전의 상관계수입니다.

In [ ]:
truth = json.load(open("data/ground_truth.json", encoding="utf-8"))
print(truth["structure"])
pd.DataFrame({"정답": pd.Series(truth["true_correlation"]),
              "움직임 제거 전": before, "움직임 제거 후": after})

## 설명해 보세요

1. 움직임을 제거하기 전과 후에 어떤 상관이 가장 크게 바뀌었나요? 왜 그렇게 되었을까요?
2. 움직임을 제거한 뒤에도 정답과 완전히 같지 않은 이유는 무엇일까요? (힌트: 잡음, 시점 수)
3. A와 B의 상관이 높다는 사실만으로 "A가 B를 움직인다"고 말할 수 있을까요?

**다른 데이터로 반복 (과제 5):** `python generate_synthetic_fmri.py --seed 7 --out data_seed7`로 다른 데이터를 만들고, 이 노트북에서 `"data/`로 시작하는 경로 네 곳을 `"data_seed7/`로 바꿔 같은 분석을 반복해 보세요. 결과는 얼마나 달라지나요? `threshold`를 0.2나 1.0으로 바꾸면 어떻게 되나요?